In [2]:
# Import Required Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import os
import cv2
import json

In [3]:
# DataFrame to store angles
angles_columns = ['frame', 'left_elbow_angle', 'left_knee_angle', 'left_shoulder_angle', 'left_hip_angle', 'left_ankle_angle',
            'right_elbow_angle', 'right_knee_angle', 'right_shoulder_angle', 'right_hip_angle', 'right_ankle_angle']

joints_columns = ['frame', 'left_elbow', 'left_knee', 'left_shoulder', 'left_hip', 'left_ankle', 'left_wrist','left_feet','right_shoulder',
                   'right_elbow', 'right_wrist', 'right_hip', 'right_knee', 'right_ankle','right_feet']


gt_joints_df = pd.DataFrame(columns=joints_columns)

gt_angles_df = pd.DataFrame(columns=angles_columns)

In [4]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Mid point
    c = np.array(c)  # End point

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle

In [16]:
" ".join(["Show","Must","Go","On"])

'Show Must Go On'

In [20]:
def get_meta_text(keypoints, keypoint_label):
    coordinates = keypoints[keypoint_label]['coordinates']
    meta_text = keypoints[keypoint_label]['meta_text']
    return coordinates + meta_text if meta_text else coordinates

def get_angle_meta_text(angle, keys, keypoints):
    meta_text = []
    for key in keys:
        if len(keypoints[key]['meta_text']) > 0:
            text = keypoints[key]['meta_text'][0]+""
            textSplit = text.split(',')
            for tempStr in textSplit:
                if tempStr not in meta_text:
                    meta_text.append(tempStr)
    meta_text = ','.join(meta_text)
    return [angle] + [meta_text] if meta_text else [angle]

# Function to process videos using MediaPipe Pose and calculate joint angles
def process_images(path, image_type):

    gt_joints_df = pd.DataFrame(columns=joints_columns)

    gt_angles_df = pd.DataFrame(columns=angles_columns)
    
    frame_files = [f for f in os.listdir(path+image_type) if f.startswith('frame_') and f.endswith('.jpg')]

    # Extract frame numbers from the filenames and sort them
    frame_numbers = sorted([int(f.split('_')[1].split('.')[0]) for f in frame_files])

    # Sort frame files based on the sorted frame numbers
    frame_files = [f"frame_{num}.jpg" for num in frame_numbers]

    outdir = f"../FINAL/groundtruth_{image_type}"
    keypointsfile = outdir+"/ls_keypoints.json"

    keypoints_data = process_json(keypointsfile)

    for i,file in enumerate(frame_files):  
        gt_angles_df, gt_joints_df = process_single_image(path+image_type+'/'+file, image_type, keypoints_data[frame_numbers[i]], os.path.join(outdir, f"frame_{frame_numbers[i]}_pose.jpg"), frame_numbers[i], gt_angles_df, gt_joints_df)

    #gt_joints_df.to_json(os.path.join(outdir, 'keypoints.json'), orient='records')
    gt_angles_df.to_json(os.path.join(outdir, 'angles.json'), orient='records')

def process_json(json_file_path):
    # Load the JSON data
    with open(json_file_path, 'r') as file:
        data = json.load(file)

    # Extract keypoints for each frame image
    keypoints_data = {}

    for item in data:
        frame_image = item['data']['frame_image']
        frame_number = int(frame_image.split('-')[-1].split('.')[0].replace('frame_', ''))
        annotations = item['annotations']
        
        for annotation in annotations:
            results = annotation['result']
            
            keypoints = {}
            for result in results:
                if result['type'] == 'keypointlabels':
                    keypoint_label = result['value']['keypointlabels'][0]
                    x = result['value']['x']
                    y = result['value']['y']
                    meta_text = result.get('meta', {}).get('text', [])
                    original_width = result['original_width']
                    original_height = result['original_height']
                    x_pixel = (x / 100) * original_width
                    y_pixel = (y / 100) * original_height
                    keypoints[keypoint_label] = {
                        'coordinates': [x_pixel, y_pixel],
                        'meta_text': meta_text
                    }
            
            keypoints_data[frame_number] = keypoints

    return keypoints_data

def process_single_image(image_file_path, image_type, keypoints, output_frame_path, frame_count, gt_angles_df, gt_joints_df):
    
    frame = cv2.imread(image_file_path)
    
    if frame is None:
        print(f"Error opening image file {image_file_path}")
        return
    
    print(f"Processing image: {image_file_path}")

    #outdir = f"../FINAL/groundtruth_{image_type}/"

    # Get coordinates for left side
    left_shoulder = keypoints['left_shoulder']['coordinates']
    left_elbow = keypoints['left_elbow']['coordinates']
    left_wrist = keypoints['left_wrist']['coordinates']
    left_hip = keypoints['left_hip']['coordinates']
    left_knee = keypoints['left_knee']['coordinates']
    left_ankle = keypoints['left_ankle']['coordinates']
    left_feet = keypoints['left_feet']['coordinates']

    right_shoulder = keypoints['right_shoulder']['coordinates']
    right_elbow = keypoints['right_elbow']['coordinates']
    right_wrist = keypoints['right_wrist']['coordinates']
    right_hip = keypoints['right_hip']['coordinates']
    right_knee = keypoints['right_knee']['coordinates']
    right_ankle = keypoints['right_ankle']['coordinates']
    right_feet = keypoints['right_feet']['coordinates']  
    
    left_shoulder_meta = get_meta_text(keypoints, 'left_shoulder')
    left_elbow_meta = get_meta_text(keypoints, 'left_elbow')
    left_wrist_meta = get_meta_text(keypoints, 'left_wrist')
    left_hip_meta = get_meta_text(keypoints, 'left_hip')
    left_knee_meta = get_meta_text(keypoints, 'left_knee')
    left_ankle_meta = get_meta_text(keypoints, 'left_ankle')
    left_feet_meta = get_meta_text(keypoints, 'left_feet')

    right_shoulder_meta = get_meta_text(keypoints, 'right_shoulder')
    right_elbow_meta = get_meta_text(keypoints, 'right_elbow')
    right_wrist_meta = get_meta_text(keypoints, 'right_wrist')
    right_hip_meta = get_meta_text(keypoints, 'right_hip')
    right_knee_meta = get_meta_text(keypoints, 'right_knee')
    right_ankle_meta = get_meta_text(keypoints, 'right_ankle')
    right_feet_meta = get_meta_text(keypoints, 'right_feet')

    # # Calculate angles for left side
    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    left_shoulder_angle = calculate_angle(left_hip, left_shoulder, left_elbow)
    left_hip_angle = calculate_angle(left_shoulder, left_hip, left_knee)
    left_ankle_angle = calculate_angle(left_knee, left_ankle, left_feet)

    # # Calculate angles for right side
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
    right_shoulder_angle = calculate_angle(right_hip, right_shoulder, right_elbow)
    right_hip_angle = calculate_angle(right_shoulder, right_hip, right_knee)
    right_ankle_angle = calculate_angle(right_knee, right_ankle, right_feet)

    #
    left_elbow_angle_meta =  get_angle_meta_text(left_elbow_angle, ['left_shoulder', 'left_elbow', 'left_wrist'], keypoints)
    left_knee_angle_meta = get_angle_meta_text(left_knee_angle, ['left_hip', 'left_knee', 'left_ankle'], keypoints)
    left_shoulder_angle_meta = get_angle_meta_text(left_shoulder_angle, ['left_hip', 'left_shoulder', 'left_elbow'], keypoints)
    left_hip_angle_meta = get_angle_meta_text(left_hip_angle, ['left_shoulder', 'left_hip', 'left_knee'], keypoints)
    left_ankle_angle_meta = get_angle_meta_text(left_ankle_angle, ['left_knee', 'left_ankle', 'left_feet'], keypoints)

    right_elbow_angle_meta = get_angle_meta_text(right_elbow_angle, ['right_shoulder', 'right_elbow', 'right_wrist'], keypoints)
    right_knee_angle_meta = get_angle_meta_text(right_knee_angle, ['right_hip', 'right_knee', 'right_ankle'], keypoints)
    right_shoulder_angle_meta = get_angle_meta_text(right_shoulder_angle, ['right_hip', 'right_shoulder', 'right_elbow'], keypoints)
    right_hip_angle_meta = get_angle_meta_text(right_hip_angle, ['right_shoulder', 'right_hip', 'right_knee'], keypoints)
    right_ankle_angle_meta = get_angle_meta_text(right_ankle_angle, ['right_knee', 'right_ankle', 'right_feet'], keypoints)

    for point in [left_shoulder, left_elbow, left_wrist, left_hip, left_knee, left_ankle, left_feet,
                right_shoulder, right_elbow, right_wrist, right_hip, right_knee, right_ankle, right_feet]:
        cv2.circle(frame, tuple(np.multiply(point, [1, 1]).astype(int)), 5, (0, 0, 255), -1)
    # Annotate angles on the image
    cv2.putText(frame, f'Left Elbow: {int(left_elbow_angle)}', tuple(np.multiply(left_elbow, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Left Knee: {int(left_knee_angle)}', tuple(np.multiply(left_knee, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Left Shoulder: {int(left_shoulder_angle)}', tuple(np.multiply(left_shoulder, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Left Hip: {int(left_hip_angle)}', tuple(np.multiply(left_hip, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Left Ankle: {int(left_ankle_angle)}', tuple(np.multiply(left_ankle, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

    cv2.putText(frame, f'Right Elbow: {int(right_elbow_angle)}', tuple(np.multiply(right_elbow, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Right Knee: {int(right_knee_angle)}', tuple(np.multiply(right_knee, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Right Shoulder: {int(right_shoulder_angle)}', tuple(np.multiply(right_shoulder, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Right Hip: {int(right_hip_angle)}', tuple(np.multiply(right_hip, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Right Ankle: {int(right_ankle_angle)}', tuple(np.multiply(right_ankle, [1, 1]).astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)

    #cv2.imwrite(output_frame_path, frame)

    # Store points into a DataFrame  

    new_joints_row = pd.DataFrame([{
        'frame': f"frame_{frame_count}",
        'left_shoulder': left_shoulder_meta,
        'left_elbow': left_elbow_meta,
        'left_wrist': left_wrist_meta,
        'left_hip': left_hip_meta,
        'left_knee': left_knee_meta,
        'left_ankle': left_ankle_meta,
        'left_feet': left_feet_meta,
        'right_shoulder': right_shoulder_meta,
        'right_elbow': right_elbow_meta,
        'right_wrist': right_wrist_meta,
        'right_hip': right_hip_meta,
        'right_knee': right_knee_meta,
        'right_ankle': right_ankle_meta,
        'right_feet': right_feet_meta
    }], columns=joints_columns)

    new_angles_row = pd.DataFrame([{
        'frame': f"frame_{frame_count}",
        'left_elbow_angle': left_elbow_angle_meta,
        'left_knee_angle': left_knee_angle_meta,
        'left_shoulder_angle': left_shoulder_angle_meta,
        'left_hip_angle': left_hip_angle_meta,
        'left_ankle_angle': left_ankle_angle_meta,
        'right_elbow_angle': right_elbow_angle_meta,
        'right_knee_angle': right_knee_angle_meta,
        'right_shoulder_angle': right_shoulder_angle_meta,
        'right_hip_angle': right_hip_angle_meta,
        'right_ankle_angle': right_ankle_angle_meta
    }], columns=angles_columns)

    gt_joints_df = pd.concat([gt_joints_df, new_joints_row], ignore_index=True)
    gt_angles_df = pd.concat([gt_angles_df, new_angles_row], ignore_index=True)

    return gt_angles_df, gt_joints_df


# Process videos for each side using MediaPipe Pose
#for side in sides:
#    process_videos_with_mediapipe(side)

process_images("../videos/", "accel")
process_images("../videos/","maxV_close")
process_images("../videos/", "maxV_far")


Processing image: ../videos/accel/frame_45.jpg
Processing image: ../videos/accel/frame_47.jpg
Processing image: ../videos/accel/frame_49.jpg
Processing image: ../videos/accel/frame_51.jpg
Processing image: ../videos/accel/frame_53.jpg
Processing image: ../videos/accel/frame_55.jpg
Processing image: ../videos/accel/frame_57.jpg
Processing image: ../videos/accel/frame_59.jpg
Processing image: ../videos/accel/frame_61.jpg
Processing image: ../videos/accel/frame_63.jpg
Processing image: ../videos/accel/frame_65.jpg
Processing image: ../videos/accel/frame_67.jpg
Processing image: ../videos/accel/frame_69.jpg
Processing image: ../videos/accel/frame_71.jpg
Processing image: ../videos/accel/frame_73.jpg
Processing image: ../videos/accel/frame_75.jpg
Processing image: ../videos/accel/frame_77.jpg
Processing image: ../videos/accel/frame_79.jpg
Processing image: ../videos/accel/frame_81.jpg
Processing image: ../videos/maxV_close/frame_6.jpg
Processing image: ../videos/maxV_close/frame_7.jpg
Proce